# Lab 04-03: RAG Deployment Pipeline

**Module:** 04 -- Assembling & Deploying Applications (22% of exam)  
**UI Sections:** Serving | Models | Catalog | SQL Editor  
**Estimated Time:** 60--75 minutes  
**Difficulty:** Advanced

---

## What and Why

This is the capstone of the RAG journey. You take your chain (Lab 04-01), your Vector
Search index (Lab 04-02), package it all with pyfunc, create a serving endpoint,
configure access control, and test it end-to-end. The exam tests the **full deployment
sequence** -- from Delta table to production endpoint.

This lab includes **strengthened coverage of endpoint access control**, which is a
critical exam topic that many candidates underestimate.

---

## Mental Model

> **Analogy:** Deploying a RAG chain is like opening a restaurant. You have perfected
> the recipe (chain), sourced ingredients (vector index), and trained the chef (pyfunc).
> Now you need a storefront (serving endpoint), a menu (API schema), a cashier
> (authentication), and health inspectors (monitoring). Skipping any of these means
> the restaurant cannot open -- or worse, opens without safety checks.

---

## Exam Alert

| Trap (Wrong) | Correct Answer |
|---|---|
| "Just deploy the model -- access control is automatic" | You must explicitly GRANT permissions on serving endpoints |
| "Serving endpoints use notebook credentials" | Endpoints use service principals or personal access tokens |
| "One serving endpoint = one model" | Endpoints can have multiple served entities with traffic splitting |
| "Access control is only for the model, not the endpoint" | BOTH the UC model AND the serving endpoint need permissions |
| "Anyone with workspace access can call any endpoint" | Endpoint access must be explicitly granted via Permissions |

---

## Learning Objectives

By the end of this lab you will be able to:

1. Describe the full RAG deployment sequence (Delta table to production endpoint)
2. Build a quick RAG chain using patterns from prior labs
3. Package the chain as a pyfunc model and register it to Unity Catalog
4. Create a Model Serving endpoint via the REST API
5. Configure endpoint access control (service principals, tokens, grants)
6. Test the deployed endpoint with curl and Python requests
7. Follow a deployment checklist for exam-ready confidence

---

## Exam Objectives Covered

| Objective | Tested Here |
|---|---|
| RAG deployment sequence | Step 1 |
| Package chain as pyfunc | Steps 2--3 |
| Register to Unity Catalog | Step 4 |
| Create serving endpoints | Step 5 |
| Endpoint access control (STRENGTHENED) | Step 6 |
| Test deployed endpoints | Step 7 |
| Full deployment checklist | Step 8 |

---

## Step 1: The Deployment Sequence Diagram

The exam frequently asks about the correct order of steps to deploy a RAG system.
Memorize this sequence:

```
+------------------+     +------------------+     +-------------------+
| 1. Delta Table   | --> | 2. Vector Search | --> | 3. RAG Chain      |
| (chunks + CDF)   |     |    Index         |     |    (retriever +   |
|                  |     |    (Delta Sync)  |     |     prompt + LLM) |
+------------------+     +------------------+     +-------------------+
                                                          |
                                                          v
+------------------+     +------------------+     +-------------------+
| 6. Client App    | <-- | 5. Serving       | <-- | 4. pyfunc Model   |
| (curl / SDK /    |     |    Endpoint      |     |    in Unity       |
|  Databricks App) |     |    + ACLs        |     |    Catalog        |
+------------------+     +------------------+     +-------------------+
```

### Each Step Depends on the Previous One

| Step | Depends On | What It Produces |
|---|---|---|
| 1. Delta Table | Raw documents, chunking pipeline | `workspace.genai_labs.document_chunks` with CDF |
| 2. Vector Search Index | Delta table (step 1) + embedding model | Searchable index for retrieval |
| 3. RAG Chain | Vector index (step 2) + LLM endpoint | Working chain in a notebook |
| 4. pyfunc + UC Registration | Chain (step 3) + MLflow | Governed model artifact |
| 5. Serving Endpoint | Registered model (step 4) | REST API for predictions |
| 6. Client App | Serving endpoint (step 5) + auth token | End-user application |

In [ ]:
# ---------------------------------------------------------------------------
# Print the deployment sequence for study reference
# ---------------------------------------------------------------------------
steps = [
    ("1", "Delta Table with CDF",
     "Chunks stored in workspace.genai_labs.document_chunks",
     "Lab 02-03"),
    ("2", "Vector Search Index",
     "Delta Sync index on the chunk table",
     "Lab 04-02"),
    ("3", "RAG Chain",
     "Retriever + prompt template + LLM call",
     "Lab 04-01"),
    ("4", "pyfunc Model in Unity Catalog",
     "Chain wrapped in PythonModel, registered to UC",
     "Lab 04-01"),
    ("5", "Serving Endpoint + Access Control",
     "REST API with auth and permissions",
     "This lab"),
    ("6", "Client Application",
     "curl, Python SDK, or Databricks App",
     "This lab"),
]

print("RAG Deployment Sequence")
print("=" * 65)
for num, name, detail, lab in steps:
    print(f"\n  Step {num}: {name}")
    print(f"           {detail}")
    print(f"           (Reference: {lab})")

print("\n" + "=" * 65)
print("EXAM TIP: The sequence is a recurring exam question.")
print("Memorize: Table --> Index --> Chain --> pyfunc --> UC --> Endpoint")


**Expected output:**

```
RAG Deployment Sequence
=================================================================

  Step 1: Delta Table with CDF
           Chunks stored in workspace.genai_labs.document_chunks
           (Reference: Lab 02-03)

  Step 2: Vector Search Index
           Delta Sync index on the chunk table
           (Reference: Lab 04-02)
  ...
```

---

## Step 2: Quick Chain Build

We reuse the RAG chain pattern from Lab 04-01: a pyfunc model with retriever, prompt
builder, and LLM call. This time, we make it deployment-ready with proper error handling
and configuration.

In [ ]:
import mlflow
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

# ---------------------------------------------------------------------------
# Configuration -- centralize all settings
# ---------------------------------------------------------------------------
CONFIG = {
    "llm_model": "databricks-meta-llama-3-3-70b-instruct",
    "embedding_model": "databricks-bge-large-en",
    "vs_endpoint": "genai_vs_endpoint",
    "vs_index": "workspace.genai_labs.document_chunks_index",
    "uc_model_name": "workspace.genai_labs.rag_chain_v1",
    "serving_endpoint_name": "genai-rag-endpoint",
    "num_retrieval_results": 3,
    "max_tokens": 512,
    "temperature": 0.1,
}

print("Deployment Configuration")
print("=" * 50)
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


**Expected output:**

```
Deployment Configuration
==================================================
  llm_model: databricks-meta-llama-3-3-70b-instruct
  embedding_model: databricks-bge-large-en
  vs_endpoint: genai_vs_endpoint
  vs_index: workspace.genai_labs.document_chunks_index
  uc_model_name: workspace.genai_labs.rag_chain_v1
  serving_endpoint_name: genai-rag-endpoint
  num_retrieval_results: 3
  max_tokens: 512
  temperature: 0.1
```

In [ ]:
# ---------------------------------------------------------------------------
# Deployment-ready RAG chain as a pyfunc model
# ---------------------------------------------------------------------------
class DeployableRAGChain(mlflow.pyfunc.PythonModel):
    """
    Production-ready RAG chain packaged as pyfunc.
    Retrieves context from Vector Search, builds a prompt, calls the LLM.
    Includes error handling suitable for a serving endpoint.
    """

    def load_context(self, context):
        """Initialize all resources. Called once when the model is loaded."""
        # In production on Databricks:
        # from databricks.vector_search.client import VectorSearchClient
        # from openai import OpenAI
        # import json
        #
        # # Load config from artifact
        # if context and context.artifacts.get("config"):
        #     with open(context.artifacts["config"]) as f:
        #         self.config = json.load(f)
        # else:
        #     self.config = CONFIG  # fallback to defaults
        #
        # # Initialize Vector Search client
        # self.vsc = VectorSearchClient()
        # self.index = self.vsc.get_index(
        #     endpoint_name=self.config["vs_endpoint"],
        #     index_name=self.config["vs_index"],
        # )
        #
        # # Initialize LLM client
        # self.client = OpenAI(
        #     base_url=f"https://{workspace_url}/serving-endpoints",
        #     api_key=os.environ.get("DATABRICKS_TOKEN"),
        # )

        self.config = CONFIG
        self.system_prompt = (
            "You are a helpful assistant. Answer based on the provided context. "
            "If the context is insufficient, say so clearly."
        )
        print(f"RAG chain initialized.")
        print(f"  LLM: {self.config['llm_model']}")
        print(f"  Index: {self.config['vs_index']}")

    def _retrieve(self, query: str) -> list:
        """Retrieve relevant chunks from Vector Search."""
        # Production:
        # try:
        #     results = self.index.similarity_search(
        #         query_text=query,
        #         columns=["chunk_text", "source_doc"],
        #         num_results=self.config["num_retrieval_results"],
        #     )
        #     return results.get("result", {}).get("data_array", [])
        # except Exception as e:
        #     print(f"Retrieval error: {e}")
        #     return []

        return [
            ["Databricks is a unified analytics platform.", "overview.txt"],
            ["Unity Catalog provides data governance.", "overview.txt"],
            ["Model Serving deploys models as REST APIs.", "serving.txt"],
        ]

    def _build_prompt(self, question: str, chunks: list) -> str:
        """Build the augmented prompt with retrieved context."""
        context = "\n---\n".join([c[0] for c in chunks])
        return (
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\n"
            f"Provide a clear, concise answer based on the context:"
        )

    def predict(self, context, model_input, params=None):
        """Main inference: retrieve context, build prompt, call LLM."""
        questions = model_input["question"].tolist()
        results = []

        for q in questions:
            try:
                # Retrieve
                chunks = self._retrieve(q)

                # Build prompt
                prompt = self._build_prompt(q, chunks)

                # Call LLM (simulated)
                # Production:
                # response = self.client.chat.completions.create(
                #     model=self.config["llm_model"],
                #     messages=[
                #         {"role": "system", "content": self.system_prompt},
                #         {"role": "user", "content": prompt},
                #     ],
                #     max_tokens=self.config["max_tokens"],
                #     temperature=self.config["temperature"],
                # )
                # answer = response.choices[0].message.content
                answer = f"[Simulated RAG answer for: {q[:50]}]"

                results.append({
                    "answer": answer,
                    "sources": [c[1] for c in chunks],
                    "num_sources": len(chunks),
                })
            except Exception as e:
                results.append({
                    "answer": f"Error processing request: {str(e)}",
                    "sources": [],
                    "num_sources": 0,
                })

        return results


# Test locally
chain = DeployableRAGChain()
chain.load_context(context=None)

test_input = pd.DataFrame({"question": [
    "What is Databricks?",
    "How does Unity Catalog work?",
]})

results = chain.predict(context=None, model_input=test_input)
for r in results:
    print(r)


**Expected output:**

```
RAG chain initialized.
  LLM: databricks-meta-llama-3-3-70b-instruct
  Index: workspace.genai_labs.document_chunks_index
{'answer': '[Simulated RAG answer for: What is Databricks?]', 'sources': ['overview.txt', 'overview.txt', 'serving.txt'], 'num_sources': 3}
{'answer': '[Simulated RAG answer for: How does Unity Catalog work?]', 'sources': ['overview.txt', 'overview.txt', 'serving.txt'], 'num_sources': 3}
```

---

## Step 3: Package as pyfunc (Reference Lab 04-01 Pattern)

In [ ]:
# ---------------------------------------------------------------------------
# Define the model signature
# ---------------------------------------------------------------------------
signature = ModelSignature(
    inputs=Schema([ColSpec("string", "question")]),
    outputs=Schema([ColSpec("string", "answer")]),
)

print("Model Signature:")
print(signature)


In [ ]:
# ---------------------------------------------------------------------------
# Log the model to MLflow
# ---------------------------------------------------------------------------
mlflow.set_experiment("04-03-rag-deployment")

with mlflow.start_run(run_name="rag-deployment-chain") as run:
    model_info = mlflow.pyfunc.log_model(
        artifact_path="rag_chain",
        python_model=DeployableRAGChain(),
        signature=signature,
        pip_requirements=[
            "mlflow>=2.12",
            "openai>=1.0",
            "databricks-vectorsearch",
            "pandas",
        ],
        input_example=pd.DataFrame({"question": ["What is Databricks?"]}),
    )

    mlflow.set_tag("deployment_stage", "staging")
    mlflow.set_tag("llm_model", CONFIG["llm_model"])
    mlflow.set_tag("vs_index", CONFIG["vs_index"])

    RUN_ID = run.info.run_id
    MODEL_URI = model_info.model_uri

    print(f"Model logged successfully.")
    print(f"  Run ID:    {RUN_ID}")
    print(f"  Model URI: {MODEL_URI}")


**Expected output:**

```
Model logged successfully.
  Run ID:    abc123def456...
  Model URI: runs:/abc123def456.../rag_chain
```

---

## Step 4: Register to Unity Catalog

In [ ]:
# ---------------------------------------------------------------------------
# Register the model to Unity Catalog
# ---------------------------------------------------------------------------
# On Databricks:
#
# mlflow.set_registry_uri("databricks-uc")
#
# result = mlflow.register_model(
#     model_uri=MODEL_URI,
#     name=CONFIG["uc_model_name"],
# )
#
# print(f"Registered: {result.name} v{result.version}")
#
# # Add aliases for deployment
# from mlflow import MlflowClient
# client = MlflowClient()
# client.set_registered_model_alias(
#     name=CONFIG["uc_model_name"],
#     alias="production",
#     version=result.version,
# )

UC_MODEL = CONFIG["uc_model_name"]

print("Unity Catalog Registration")
print("=" * 50)
print()
print(f"Model: {UC_MODEL}")
print(f"Source: {MODEL_URI}")
print()
print("Commands (run on Databricks):")
print()
print('  mlflow.set_registry_uri("databricks-uc")')
print(f'  result = mlflow.register_model(')
print(f'      model_uri="{MODEL_URI}",')
print(f'      name="{UC_MODEL}",')
print(f'  )')
print()
print("  # Set alias for serving")
print(f'  client.set_registered_model_alias(')
print(f'      name="{UC_MODEL}",')
print(f'      alias="production",')
print(f'      version=result.version,')
print(f'  )')
print()
print("UI Navigation:")
print("  **UI -->** Left nav --> **Models** --> search for 'rag_chain_v1'")
print("  Click the model --> Versions tab --> see version 1 with 'production' alias")


**Expected output:**

```
Unity Catalog Registration
==================================================

Model: workspace.genai_labs.rag_chain_v1
Source: runs:/abc123.../rag_chain

Commands (run on Databricks):

  mlflow.set_registry_uri("databricks-uc")
  result = mlflow.register_model(...)
```

### Model Aliases vs. Version Numbers

| Concept | Example | Use Case |
|---|---|---|
| Version number | `v1`, `v2`, `v3` | Immutable identifier for each registration |
| Alias | `"production"`, `"staging"` | Mutable pointer to a version -- use for serving |

---

## Step 5: Create a Serving Endpoint

A serving endpoint wraps your registered model in a REST API. You can create it via:
1. **UI**: **UI -->** Left nav --> **Serving** --> **Create serving endpoint**
2. **REST API**: `POST /api/2.0/serving-endpoints`
3. **Databricks SDK**: `w.serving_endpoints.create()`

### Endpoint Configuration

| Setting | Purpose | Example |
|---|---|---|
| `name` | Unique endpoint identifier | `"genai-rag-endpoint"` |
| `served_entities` | Which model(s) to serve | Model name + version/alias |
| `workload_size` | Compute size per container | `"Small"`, `"Medium"`, `"Large"` |
| `scale_to_zero_enabled` | Scale down when idle | `true` (saves cost) |
| `workload_type` | GPU vs. CPU | `"GPU_SMALL"` for LLM inference |

In [ ]:
# ---------------------------------------------------------------------------
# Create a serving endpoint via REST API
# ---------------------------------------------------------------------------
import json

ENDPOINT_NAME = CONFIG["serving_endpoint_name"]

# The JSON payload for creating a serving endpoint
endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "name": "rag-chain-entity",
                "entity_name": CONFIG["uc_model_name"],
                "entity_version": "1",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            }
        ],
        "auto_capture_config": {
            "catalog_name": "main",
            "schema_name": "genai_labs",
            "table_name_prefix": "rag_endpoint",
        },
    },
}

print("Serving Endpoint Configuration")
print("=" * 50)
print(json.dumps(endpoint_config, indent=2))


**Expected output:**

```json
{
  "name": "genai-rag-endpoint",
  "config": {
    "served_entities": [
      {
        "name": "rag-chain-entity",
        "entity_name": "workspace.genai_labs.rag_chain_v1",
        "entity_version": "1",
        "workload_size": "Small",
        "scale_to_zero_enabled": true
      }
    ],
    "auto_capture_config": {
      "catalog_name": "main",
      "schema_name": "genai_labs",
      "table_name_prefix": "rag_endpoint"
    }
  }
}
```

### auto_capture_config: Inference Tables

The `auto_capture_config` section enables **inference tables** -- Delta tables that
automatically log every request and response to the endpoint. This is critical for:
- **Monitoring**: track latency, error rates, usage patterns
- **Evaluation**: compare model quality over time
- **Debugging**: replay failed requests
- **Compliance**: audit trail of all predictions

In [ ]:
# ---------------------------------------------------------------------------
# Create the endpoint (REST API call)
# ---------------------------------------------------------------------------
# On Databricks:
#
# import requests
#
# WORKSPACE_URL = "https://<your-workspace>.cloud.databricks.com"
# TOKEN = dbutils.secrets.get(scope="genai", key="token")
#
# response = requests.post(
#     f"{WORKSPACE_URL}/api/2.0/serving-endpoints",
#     headers={"Authorization": f"Bearer {TOKEN}"},
#     json=endpoint_config,
# )
#
# if response.status_code == 200:
#     print(f"Endpoint '{ENDPOINT_NAME}' created successfully.")
#     print(f"State: {response.json()['state']['ready']}")
# else:
#     print(f"Error: {response.status_code}")
#     print(response.json())
#
# Alternative: Databricks SDK
# from databricks.sdk import WorkspaceClient
# w = WorkspaceClient()
# w.serving_endpoints.create(
#     name=ENDPOINT_NAME,
#     config=endpoint_config["config"],
# )

print("Creating Serving Endpoint")
print("=" * 50)
print()
print(f"POST /api/2.0/serving-endpoints")
print(f"Body: (see endpoint_config above)")
print()
print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Model: {CONFIG['uc_model_name']} v1")
print(f"Status: PENDING --> READY (takes 5-15 minutes)")
print()
print("UI Navigation:")
print("  **UI -->** Left nav --> **Serving** --> find 'genai-rag-endpoint'")
print("  Status will show: Pending, then Ready")
print("  Click the endpoint to see: served entities, compute, logs")


**Expected output:**

```
Creating Serving Endpoint
==================================================

POST /api/2.0/serving-endpoints
Body: (see endpoint_config above)

Endpoint: genai-rag-endpoint
Model: workspace.genai_labs.rag_chain_v1 v1
Status: PENDING --> READY (takes 5-15 minutes)
```

In [ ]:
# ---------------------------------------------------------------------------
# Traffic splitting: multiple served entities on one endpoint
# ---------------------------------------------------------------------------
# An endpoint can serve multiple model versions with traffic splitting.
# This enables A/B testing and gradual rollouts.

traffic_split_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "name": "rag-chain-v1",
                "entity_name": CONFIG["uc_model_name"],
                "entity_version": "1",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            },
            {
                "name": "rag-chain-v2",
                "entity_name": CONFIG["uc_model_name"],
                "entity_version": "2",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            },
        ],
        "traffic_config": {
            "routes": [
                {"served_model_name": "rag-chain-v1", "traffic_percentage": 90},
                {"served_model_name": "rag-chain-v2", "traffic_percentage": 10},
            ]
        },
    },
}

print("Traffic Splitting Configuration")
print("=" * 50)
print()
print("  rag-chain-v1: 90% of traffic (current production)")
print("  rag-chain-v2: 10% of traffic (canary/experiment)")
print()
print("Use case: A/B test a new model version before full rollout.")
print("Exam trap: 'One endpoint = one model' is WRONG.")
print("Correct: Endpoints support multiple served entities with traffic splitting.")


**Expected output:**

```
Traffic Splitting Configuration
==================================================

  rag-chain-v1: 90% of traffic (current production)
  rag-chain-v2: 10% of traffic (canary/experiment)

Use case: A/B test a new model version before full rollout.
Exam trap: 'One endpoint = one model' is WRONG.
Correct: Endpoints support multiple served entities with traffic splitting.
```

---

## Step 6: Endpoint Access Control (STRENGTHENED)

This is one of the most underestimated exam topics. Access control for serving endpoints
involves multiple layers: who can manage the endpoint, who can query it, and what
credentials the endpoint itself uses to access other resources.

### The Three Layers of Access Control

```
Layer 1: Who can MANAGE the endpoint?
  (create, update, delete, view config)
  --> Workspace permissions + endpoint-level permissions

Layer 2: Who can QUERY the endpoint?
  (send prediction requests)
  --> GRANT QUERY on the serving endpoint

Layer 3: What can the ENDPOINT ACCESS?
  (which models, tables, indexes does the endpoint need)
  --> Service principal permissions on UC resources
```

In [ ]:
# ---------------------------------------------------------------------------
# Layer 1: Endpoint management permissions
# ---------------------------------------------------------------------------
print("Layer 1: Endpoint Management Permissions")
print("=" * 55)
print()
print("Who can create, update, or delete serving endpoints?")
print()
print("  Permission levels:")
print("  +-----------------+-------------------------------------------+")
print("  | Level           | Can Do                                    |")
print("  +-----------------+-------------------------------------------+")
print("  | CAN_MANAGE      | Full control (create, update, delete)     |")
print("  | CAN_QUERY       | Send prediction requests only             |")
print("  | CAN_VIEW        | View endpoint config (read-only)          |")
print("  +-----------------+-------------------------------------------+")
print()
print("UI Navigation:")
print("  **UI -->** Serving --> click endpoint --> **Permissions** tab")
print("  Add users, groups, or service principals with appropriate level.")


**Expected output:**

```
Layer 1: Endpoint Management Permissions
=======================================================

Who can create, update, or delete serving endpoints?

  Permission levels:
  +-----------------+-------------------------------------------+
  | Level           | Can Do                                    |
  +-----------------+-------------------------------------------+
  | CAN_MANAGE      | Full control (create, update, delete)     |
  | CAN_QUERY       | Send prediction requests only             |
  | CAN_VIEW        | View endpoint config (read-only)          |
  +-----------------+-------------------------------------------+
```

In [ ]:
# ---------------------------------------------------------------------------
# Layer 2: Query permissions -- who can call the endpoint
# ---------------------------------------------------------------------------
print("Layer 2: Query Permissions (GRANT QUERY)")
print("=" * 55)
print()
print("By default, only the endpoint creator can query it.")
print("You must explicitly GRANT query access to other users/groups.")
print()
print("Method 1: SQL GRANT syntax")
print("-" * 40)
print()
print("  -- Grant query access to a user")
print(f"  GRANT QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("  TO `user@company.com`;")
print()
print("  -- Grant query access to a group")
print(f"  GRANT QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("  TO `data-science-team`;")
print()
print("  -- Grant query access to a service principal")
print(f"  GRANT QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("  TO `rag-app-service-principal`;")
print()
print("  -- Revoke access")
print(f"  REVOKE QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("  FROM `user@company.com`;")
print()
print("Method 2: REST API")
print("-" * 40)
print()
print(f"  PUT /api/2.0/permissions/serving-endpoints/{ENDPOINT_NAME}")
print('  Body: {"access_control_list": [{"user_name": "...", "permission_level": "CAN_QUERY"}]}')
print()
print("Method 3: UI")
print("-" * 40)
print("  **UI -->** Serving --> click endpoint --> **Permissions** tab --> **Add**")
print("  Select user/group/SP --> set permission level --> **Save**")


**Expected output:**

```
Layer 2: Query Permissions (GRANT QUERY)
=======================================================

By default, only the endpoint creator can query it.
You must explicitly GRANT query access to other users/groups.

Method 1: SQL GRANT syntax
----------------------------------------

  GRANT QUERY ON SERVING_ENDPOINT `genai-rag-endpoint`
  TO `user@company.com`;
  ...
```

> **Analogy:** The endpoint is like a locked restaurant kitchen. Just because someone
> has a key to the building (workspace access) does not mean they can walk into the
> kitchen (query the endpoint). You must give them a kitchen key (GRANT QUERY)
> specifically.

In [ ]:
# ---------------------------------------------------------------------------
# Layer 3: What the endpoint itself can access (service principals)
# ---------------------------------------------------------------------------
print("Layer 3: Endpoint Resource Access (Service Principals)")
print("=" * 60)
print()
print("When a serving endpoint runs your pyfunc model, it needs access to:")
print("  - The registered model in Unity Catalog")
print("  - The Vector Search index (for retrieval)")
print("  - The Foundation Model API endpoint (for LLM calls)")
print("  - Any secrets or tokens referenced in the model")
print()
print("Who runs the code inside the endpoint?")
print("-" * 50)
print()
print("  Option A: Service Principal (RECOMMENDED)")
print("    - A non-human identity created for the application")
print("    - Has its own permissions, independent of any user")
print("    - Survives when employees leave the company")
print("    - Best practice for production deployments")
print()
print("  Option B: Endpoint Creator's Identity")
print("    - The endpoint runs with the creator's permissions")
print("    - If the creator loses access, the endpoint breaks")
print("    - Acceptable for development/testing only")
print()
print("Granting the service principal access to resources:")
print("-" * 50)
print()
print("  -- Grant access to the UC model")
print(f"  GRANT EXECUTE ON FUNCTION {CONFIG['uc_model_name']}")
print("  TO `rag-service-principal`;")
print()
print("  -- Grant access to the Vector Search index")
print(f"  GRANT SELECT ON TABLE {CONFIG['vs_index']}")
print("  TO `rag-service-principal`;")
print()
print("  -- Grant access to the source Delta table")
print("  GRANT SELECT ON TABLE workspace.genai_labs.document_chunks")
print("  TO `rag-service-principal`;")


**Expected output:**

```
Layer 3: Endpoint Resource Access (Service Principals)
============================================================

When a serving endpoint runs your pyfunc model, it needs access to:
  - The registered model in Unity Catalog
  - The Vector Search index (for retrieval)
  - The Foundation Model API endpoint (for LLM calls)
  ...
```

### Service Principals: What They Are

| Concept | Human User | Service Principal |
|---|---|---|
| Identity | email address | application ID |
| Authentication | username/password, SSO | client ID + client secret (OAuth) or PAT |
| Lifecycle | tied to employment | tied to application |
| Best for | interactive use | automated systems, endpoints |
| Exam keyword | "user" | "service principal" or "non-human identity" |

In [ ]:
# ---------------------------------------------------------------------------
# Authentication methods for calling the endpoint
# ---------------------------------------------------------------------------
print("Authentication Methods for Endpoint Calls")
print("=" * 55)
print()
print("1. Personal Access Token (PAT)")
print("-" * 40)
print("  - Tied to a specific user")
print("  - Created in: **UI -->** User Settings --> Developer --> Access Tokens")
print("  - Usage: Authorization: Bearer <PAT>")
print("  - Risks: expires, tied to one person, overly broad permissions")
print("  - Best for: development and testing")
print()
print("2. Service Principal Token")
print("-" * 40)
print("  - Tied to a service principal (non-human identity)")
print("  - Created via OAuth: client_id + client_secret --> access_token")
print("  - Usage: Authorization: Bearer <SP_TOKEN>")
print("  - Benefits: not tied to a person, scoped permissions, rotatable")
print("  - Best for: production applications")
print()
print("3. OAuth Machine-to-Machine (M2M) Token")
print("-" * 40)
print("  - Generated via OAuth 2.0 client credentials flow")
print("  - Short-lived, auto-renewable")
print("  - Most secure option for automated systems")
print()
print("Exam Tip: Know the difference between PAT and SP token.")
print("  PAT = personal, human-tied, for dev")
print("  SP token = application-tied, for production")


**Expected output:**

```
Authentication Methods for Endpoint Calls
=======================================================

1. Personal Access Token (PAT)
----------------------------------------
  - Tied to a specific user
  - Created in: **UI -->** User Settings --> Developer --> Access Tokens
  ...
```

In [ ]:
# ---------------------------------------------------------------------------
# Credential passthrough and environment variables
# ---------------------------------------------------------------------------
print("Credential Passthrough")
print("=" * 55)
print()
print("What is credential passthrough?")
print("  When the endpoint uses the CALLER's identity to access")
print("  downstream resources (like Vector Search or Delta tables).")
print()
print("How it works:")
print("  1. User A calls the endpoint with their token")
print("  2. The endpoint's pyfunc model accesses Vector Search")
print("  3. Vector Search checks User A's permissions (not the endpoint's)")
print("  4. If User A has SELECT on the index, the query succeeds")
print()
print("When to use credential passthrough:")
print("  - Multi-tenant applications where each user should only")
print("    access their own data")
print("  - Compliance scenarios requiring user-level audit trails")
print()
print("When NOT to use credential passthrough:")
print("  - Shared endpoints where all users should see the same data")
print("  - When you want the endpoint to have its own identity (use SP)")
print()
print("Environment variables in endpoints:")
print("-" * 40)
print("  Endpoints can access Databricks secrets via environment variables:")
print("  - DATABRICKS_TOKEN: auto-injected, used for API calls")
print("  - Custom secrets: configured via endpoint environment variables")
print()
print("  # In endpoint config:")
print('  "environment_vars": {')
print('      "API_KEY": "{{secrets/genai/external_api_key}}"')
print('  }')


**Expected output:**

```
Credential Passthrough
=======================================================

What is credential passthrough?
  When the endpoint uses the CALLER's identity to access
  downstream resources (like Vector Search or Delta tables).
  ...
```

In [ ]:
# ---------------------------------------------------------------------------
# Complete access control setup example
# ---------------------------------------------------------------------------
print("Complete Access Control Setup for a RAG Endpoint")
print("=" * 60)
print()
print("Step-by-step (on Databricks):")
print()
print("1. Create a service principal for the endpoint:")
print("   **UI -->** Admin Console --> Service Principals --> Add")
print('   Name: "rag-endpoint-sp"')
print()
print("2. Grant the SP access to required UC resources:")
print("   -- SQL Editor or notebook:")
print(f"   GRANT EXECUTE ON FUNCTION {CONFIG['uc_model_name']}")
print("   TO `rag-endpoint-sp`;")
print()
print("   GRANT SELECT ON TABLE workspace.genai_labs.document_chunks")
print("   TO `rag-endpoint-sp`;")
print()
print("3. Create the endpoint (see Step 5)")
print()
print("4. Grant QUERY access to the endpoint consumers:")
print(f"   GRANT QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("   TO `app-frontend-sp`;")
print()
print(f"   GRANT QUERY ON SERVING_ENDPOINT `{ENDPOINT_NAME}`")
print("   TO `data-science-team`;")
print()
print("5. Verify permissions:")
print(f"   SHOW GRANTS ON SERVING_ENDPOINT `{ENDPOINT_NAME}`;")


**Expected output:**

```
Complete Access Control Setup for a RAG Endpoint
============================================================

Step-by-step (on Databricks):

1. Create a service principal for the endpoint:
   ...
2. Grant the SP access to required UC resources:
   ...
3. Create the endpoint
4. Grant QUERY access to endpoint consumers:
   ...
5. Verify permissions:
   SHOW GRANTS ON SERVING_ENDPOINT `genai-rag-endpoint`;
```

### Access Control Decision Matrix

| Scenario | Auth Method | Endpoint Permission | Resource Permission |
|---|---|---|---|
| Developer testing locally | PAT | Creator (automatic) | User's own UC grants |
| Production app backend | SP token (OAuth) | GRANT QUERY to SP | SP has UC grants |
| Data science team exploring | Group PAT or SSO | GRANT QUERY to group | Group has UC grants |
| Multi-tenant app | Credential passthrough | GRANT QUERY to all users | Per-user UC grants |

### UI Walkthrough: Permissions Tab

**UI -->** Serving --> click `genai-rag-endpoint` --> **Permissions** tab

You will see:
- **Owner**: the user who created the endpoint (has CAN_MANAGE)
- **Access Control List**: table of users/groups/SPs with their permission levels
- **Add** button: add new entries with CAN_MANAGE, CAN_QUERY, or CAN_VIEW

In [ ]:
# ---------------------------------------------------------------------------
# Exam practice: access control scenarios
# ---------------------------------------------------------------------------
acl_scenarios = [
    {
        "question": "A new data scientist joins the team and needs to test the\n"
                    "RAG endpoint. What permissions do they need?",
        "answer": "GRANT QUERY on the serving endpoint. They do NOT need\n"
                  "CAN_MANAGE unless they need to modify endpoint config.",
    },
    {
        "question": "Your production app's service principal can call the endpoint\n"
                    "but gets 'permission denied' when the model queries Vector Search.\n"
                    "What is missing?",
        "answer": "The service principal needs SELECT permission on the Vector\n"
                  "Search index (or the underlying Delta table). Endpoint QUERY\n"
                  "permission alone is not enough -- the SP also needs access to\n"
                  "downstream resources.",
    },
    {
        "question": "The endpoint was created by an employee who just left the\n"
                    "company. The endpoint stops working. Why?",
        "answer": "The endpoint was using the creator's credentials (not a service\n"
                  "principal). When the employee's account is deactivated, the\n"
                  "endpoint loses access. Solution: use a service principal.",
    },
    {
        "question": "You want all users to be able to query the endpoint, but each\n"
                    "user should only retrieve documents they have access to. How?",
        "answer": "Use credential passthrough. The endpoint uses each caller's\n"
                  "identity to query Vector Search, so UC permissions are enforced\n"
                  "per-user.",
    },
]

print("Exam Practice: Access Control Scenarios")
print("=" * 60)

for i, s in enumerate(acl_scenarios, 1):
    print(f"\nQ{i}: {s['question']}")
    print(f"A{i}: {s['answer']}")


**Expected output:**

```
Exam Practice: Access Control Scenarios
============================================================

Q1: A new data scientist joins the team and needs to test the
    RAG endpoint. What permissions do they need?
A1: GRANT QUERY on the serving endpoint. They do NOT need
    CAN_MANAGE unless they need to modify endpoint config.

Q2: Your production app's service principal can call the endpoint
    but gets 'permission denied' when the model queries Vector Search.
    What is missing?
A2: The service principal needs SELECT permission on the Vector
    Search index (or the underlying Delta table)...
```

---

## Step 7: Testing the Endpoint

Once the endpoint is READY, you can test it with curl, Python requests, or the
Databricks SDK.

In [ ]:
# ---------------------------------------------------------------------------
# Testing with curl
# ---------------------------------------------------------------------------
WORKSPACE_URL = "https://<your-workspace>.cloud.databricks.com"

curl_command = f"""curl -X POST \\
  {WORKSPACE_URL}/serving-endpoints/{ENDPOINT_NAME}/invocations \\
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "dataframe_split": {{
      "columns": ["question"],
      "data": [["What is Databricks Unity Catalog?"]]
    }}
  }}'"""

print("Testing with curl")
print("=" * 50)
print()
print(curl_command)
print()
print("Expected response:")
print(json.dumps({
    "predictions": [
        {
            "answer": "Unity Catalog is Databricks' governance layer...",
            "sources": ["overview.txt", "overview.txt"],
            "num_sources": 2,
        }
    ]
}, indent=2))


**Expected output:**

```
Testing with curl
==================================================

curl -X POST \
  https://<your-workspace>.cloud.databricks.com/serving-endpoints/genai-rag-endpoint/invocations \
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \
  -H "Content-Type: application/json" \
  -d '...'

Expected response:
{
  "predictions": [
    {
      "answer": "Unity Catalog is Databricks' governance layer...",
      "sources": ["overview.txt", "overview.txt"],
      "num_sources": 2
    }
  ]
}
```

In [ ]:
# ---------------------------------------------------------------------------
# Testing with Python requests
# ---------------------------------------------------------------------------
# On Databricks or any Python environment:
#
# import requests
# import os
#
# WORKSPACE_URL = "https://<workspace>.cloud.databricks.com"
# TOKEN = os.environ.get("DATABRICKS_TOKEN")
#
# response = requests.post(
#     f"{WORKSPACE_URL}/serving-endpoints/{ENDPOINT_NAME}/invocations",
#     headers={
#         "Authorization": f"Bearer {TOKEN}",
#         "Content-Type": "application/json",
#     },
#     json={
#         "dataframe_split": {
#             "columns": ["question"],
#             "data": [["What is Databricks?"]],
#         }
#     },
# )
#
# print(f"Status: {response.status_code}")
# print(f"Response: {response.json()}")

print("Testing with Python requests")
print("=" * 50)
print()
print("import requests")
print("import os")
print()
print("response = requests.post(")
print(f'    f"{{WORKSPACE_URL}}/serving-endpoints/{ENDPOINT_NAME}/invocations",')
print('    headers={"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"},')
print('    json={"dataframe_split": {"columns": ["question"], "data": [["What is Databricks?"]]}},')
print(")")
print()
print("Simulated response:")
print(f"  Status: 200")
print(f"  Body: {{\"predictions\": [{{\"answer\": \"Databricks is...\", ...}}]}}")


**Expected output:**

```
Testing with Python requests
==================================================

import requests
import os

response = requests.post(
    ...
)

Simulated response:
  Status: 200
  Body: {"predictions": [{"answer": "Databricks is...", ...}]}
```

### Request Format: dataframe_split vs. dataframe_records

| Format | Structure | When to Use |
|---|---|---|
| `dataframe_split` | `{"columns": [...], "data": [[...]]}` | Default for pyfunc models |
| `dataframe_records` | `[{"col": "val"}, ...]` | Alternative row-oriented format |
| `instances` | `[{"col": "val"}, ...]` | TensorFlow-style format |

In [ ]:
# ---------------------------------------------------------------------------
# Testing with Databricks SDK
# ---------------------------------------------------------------------------
# from databricks.sdk import WorkspaceClient
#
# w = WorkspaceClient()
#
# response = w.serving_endpoints.query(
#     name=ENDPOINT_NAME,
#     dataframe_split={
#         "columns": ["question"],
#         "data": [["What is Databricks?"]],
#     },
# )
#
# print(response.predictions)

print("Testing with Databricks SDK")
print("=" * 50)
print()
print("from databricks.sdk import WorkspaceClient")
print()
print("w = WorkspaceClient()")
print(f'response = w.serving_endpoints.query(')
print(f'    name="{ENDPOINT_NAME}",')
print(f'    dataframe_split={{"columns": ["question"], "data": [["What is Databricks?"]]}},')
print(f')')
print()
print("Benefits of the SDK:")
print("  - Automatic authentication (uses workspace config)")
print("  - Type-safe response objects")
print("  - Built-in retry logic")


**Expected output:**

```
Testing with Databricks SDK
==================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
response = w.serving_endpoints.query(
    name="genai-rag-endpoint",
    ...
)
```

In [ ]:
# ---------------------------------------------------------------------------
# Error handling: common endpoint errors and fixes
# ---------------------------------------------------------------------------
errors = [
    ("401 Unauthorized",
     "Invalid or expired token",
     "Refresh your PAT or SP token. Check token scope."),
    ("403 Forbidden",
     "Caller does not have QUERY permission on the endpoint",
     "GRANT QUERY ON SERVING_ENDPOINT `name` TO `user`;"),
    ("404 Not Found",
     "Endpoint does not exist or wrong URL",
     "Check endpoint name and workspace URL."),
    ("422 Unprocessable Entity",
     "Input does not match model signature",
     "Check column names and types against the ModelSignature."),
    ("500 Internal Server Error",
     "Model code threw an exception",
     "Check endpoint logs for stack trace. Fix predict() method."),
    ("503 Service Unavailable",
     "Endpoint is scaling up from zero",
     "Wait and retry. Set scale_to_zero_enabled=false for always-on."),
]

print("Common Endpoint Errors")
print("=" * 65)
print(f"{'Status':<10} {'Cause':<45} {'Fix'}")
print("-" * 65)
for status, cause, fix in errors:
    print(f"{status:<10} {cause:<45} {fix[:50]}")


**Expected output:**

```
Common Endpoint Errors
=================================================================
Status     Cause                                          Fix
-----------------------------------------------------------------
401 Unauthorized Invalid or expired token                   Refresh your PAT or SP token...
403 Forbidden    Caller does not have QUERY permission      GRANT QUERY ON SERVING_ENDPOINT...
404 Not Found    Endpoint does not exist or wrong URL       Check endpoint name and workspace URL.
...
```

---

## Step 8: The Full Deployment Checklist (Exam-Ready Summary)

Use this checklist to verify your RAG deployment is complete and exam-ready.

In [ ]:
# ---------------------------------------------------------------------------
# Full deployment checklist
# ---------------------------------------------------------------------------
checklist = [
    ("Data Layer", [
        "Delta table exists in Unity Catalog (workspace.genai_labs.document_chunks)",
        "Change Data Feed is enabled (delta.enableChangeDataFeed = true)",
        "Table has correct schema: chunk_id, chunk_text, source_doc, etc.",
        "Data is populated and verified with SELECT queries",
    ]),
    ("Retrieval Layer", [
        "Vector Search endpoint is ONLINE (genai_vs_endpoint)",
        "Delta Sync Index is created and synced (document_chunks_index)",
        "Embedding model endpoint is specified (databricks-bge-large-en)",
        "similarity_search() returns relevant results",
    ]),
    ("Model Layer", [
        "RAG chain is wrapped in mlflow.pyfunc.PythonModel",
        "ModelSignature is defined with ColSpec (inputs + outputs)",
        "Model is logged with mlflow.pyfunc.log_model()",
        "pip_requirements includes all dependencies",
        "Model is tested locally with load_model() + predict()",
    ]),
    ("Registry Layer", [
        "Model is registered to Unity Catalog (workspace.genai_labs.rag_chain_v1)",
        "Model version has a 'production' alias",
        "Model description and tags are set",
    ]),
    ("Serving Layer", [
        "Serving endpoint is created and READY (genai-rag-endpoint)",
        "served_entities points to the correct model version/alias",
        "auto_capture_config enables inference tables",
        "Endpoint responds to test requests (200 OK)",
    ]),
    ("Access Control Layer", [
        "Service principal is created for the endpoint",
        "SP has EXECUTE on the UC model",
        "SP has SELECT on the Vector Search index",
        "GRANT QUERY on endpoint for all consumer identities",
        "Authentication method chosen (PAT for dev, SP token for prod)",
        "SHOW GRANTS confirms correct permissions",
    ]),
]

print("RAG Deployment Checklist")
print("=" * 65)
for section, items in checklist:
    print(f"\n  [{section}]")
    for item in items:
        print(f"    [ ] {item}")
print("\n" + "=" * 65)
print("All boxes checked? You are exam-ready for Module 04.")


**Expected output:**

```
RAG Deployment Checklist
=================================================================

  [Data Layer]
    [ ] Delta table exists in Unity Catalog
    [ ] Change Data Feed is enabled
    ...

  [Retrieval Layer]
    [ ] Vector Search endpoint is ONLINE
    ...

  [Access Control Layer]
    [ ] Service principal is created for the endpoint
    [ ] SP has EXECUTE on the UC model
    [ ] SP has SELECT on the Vector Search index
    [ ] GRANT QUERY on endpoint for all consumer identities
    ...
=================================================================
All boxes checked? You are exam-ready for Module 04.
```

---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 1 | The deployment sequence is: Table --> Index --> Chain --> pyfunc --> UC --> Endpoint | Recurring exam question |
| 2 | Serving endpoints need EXPLICIT permission grants -- not automatic | Most common trap on access control |
| 3 | Use service principals (not personal tokens) for production endpoints | SP survives employee turnover |
| 4 | GRANT QUERY ON SERVING_ENDPOINT is the SQL syntax for query access | Know the exact syntax |
| 5 | One endpoint can serve MULTIPLE model versions with traffic splitting | Trap: "one endpoint = one model" |
| 6 | auto_capture_config enables inference tables (request/response logging) | Know what inference tables are |
| 7 | The endpoint's pyfunc model needs UC permissions for downstream resources | SP needs SELECT on indexes/tables |
| 8 | Credential passthrough uses the caller's identity, not the endpoint's | Know when to use vs. SP identity |
| 9 | 403 errors mean missing QUERY permission; 401 means invalid token | Troubleshooting access issues |
| 10 | Model aliases ("production") are mutable pointers; version numbers are not | Know how aliases work for serving |

---

## Key Takeaways

### The Deployment Pipeline
- RAG deployment follows a strict sequence: Delta table (with CDF) --> Vector Search
  index --> RAG chain --> pyfunc model --> Unity Catalog --> Serving endpoint
- Each step depends on the previous one being complete
- The exam tests this sequence repeatedly

### Serving Endpoints
- Created via REST API, SDK, or UI
- Support multiple served entities with traffic splitting
- auto_capture_config enables inference tables for monitoring
- Scale-to-zero saves cost but adds cold-start latency

### Access Control (CRITICAL FOR EXAM)
- **Three layers**: endpoint management, query access, resource access
- **GRANT QUERY ON SERVING_ENDPOINT** is the SQL syntax for query access
- **Service principals** are the recommended identity for production endpoints
- **PATs** are for development only; **SP tokens** (OAuth) are for production
- **Credential passthrough** uses the caller's identity for downstream access
- The endpoint's service principal needs permissions on UC models, indexes, and tables

### Testing
- Use curl, Python requests, or Databricks SDK to test endpoints
- Request format: `dataframe_split` with columns and data arrays
- Know common error codes: 401 (auth), 403 (permission), 422 (schema), 503 (scaling)

---

## Next Lab

**Lab 04-04: Foundation Model APIs** -- Now that you have a deployed RAG endpoint, learn
how Databricks Foundation Model APIs provide the LLM backbone (pay-per-token,
provisioned throughput, and external models via AI Gateway).